# 05 — Adaptive Exam Engine

This notebook demonstrates the **second LangGraph agent** in CertOps: an adaptive certification
exam that consumes saved program artifacts and delivers a conversational assessment.

The exam engine:
1. Loads a saved program's item bank, rubrics, and competency framework
2. Adaptively selects items — prioritising untested and weak domains
3. Presents each item conversationally and waits for the learner's response
4. Evaluates responses using LLM-as-judge against rubrics and model answers
5. Optionally probes with a follow-up question when the evaluation is borderline
6. Tracks per-domain proficiency and determines pass/fail

## Key Concepts
- **`interrupt()`**: Pauses the graph to wait for learner input, then resumes via `Command(resume=...)`
- **LLM-as-Judge**: GPT-4o scores free-text responses against rubric criteria with structured output
- **Adaptive routing**: Conditional edges select the next item based on domain proficiency gaps
- **Checkpointing**: `MemorySaver` persists exam state so sessions can survive interruptions

## 1. Setup

In [1]:
import json
import os
import random
from pathlib import Path
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

load_dotenv(Path("../.env"))
print("Imports loaded.")

Imports loaded.


## 2. Schemas and State

The exam state tracks everything: the loaded program data, current item, evaluation results,
per-domain proficiency, and the final pass/fail outcome.

In [2]:
import operator
from typing import Annotated, Optional, TypedDict
from pydantic import BaseModel, Field


class CriterionScore(BaseModel):
    criterion: str
    score: int = Field(ge=1, le=3)
    justification: str


class EvaluationResult(BaseModel):
    criterion_scores: list[CriterionScore]
    weighted_score: float
    confidence: str = Field(description="'clear' or 'borderline'")
    feedback: str
    probe_question: str | None = None


class ExamResultSummary(BaseModel):
    passed: bool
    overall_score: float
    summary: str
    recommendation: str


DIFFICULTY_MAP = {"analysis": 1, "scenario": 2, "performance": 3}


class ExamState(TypedDict):
    program_id: str
    learner_id: str
    program_name: str
    item_bank: list[dict]
    rubrics: list[dict]
    framework: dict
    assessments: list[dict]
    items_remaining: list[dict]
    domain_rubrics: dict
    current_item: Optional[dict]
    current_domain: str
    current_evaluation: Optional[dict]
    probed: bool
    messages: Annotated[list[dict], operator.add]
    items_administered: Annotated[list[dict], operator.add]
    domain_proficiency: dict
    exam_complete: bool
    passed: Optional[bool]
    result_summary: Optional[dict]


print("Schemas defined.")

Schemas defined.


## 3. Load Exemplar Data

We use the AI Champion exemplar as our test program.

In [3]:
data_path = Path("../data/certops_ai_champion_output.json")
program_data = json.loads(data_path.read_text())

print(f"Loaded program with {len(program_data['item_bank'])} items, "
      f"{len(program_data['rubrics'])} rubrics, "
      f"{len(program_data['competency_framework']['domains'])} domains")

Loaded program with 10 items, 4 rubrics, 4 domains


## 4. Node Functions

Seven nodes form the exam pipeline. `present_item` and `probe_or_score` use `interrupt()`
to pause for learner input.

In [4]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)


def _extract_domain(competency_ref: str) -> str:
    return competency_ref.split(":")[0].strip()


def _build_domain_rubrics(rubrics, assessments):
    assessment_domain = {}
    for a in assessments:
        domain = _extract_domain(a.get("competency_ref", ""))
        assessment_domain[a["title"]] = domain
    domain_rubrics = {}
    for r in rubrics:
        domain = assessment_domain.get(r["assessment_ref"])
        if domain and domain not in domain_rubrics:
            domain_rubrics[domain] = r["criteria"]
    return domain_rubrics


def _format_rubric_criteria(criteria):
    return "\n".join(
        f"- {c['criterion']} (weight {c['weight']})\n"
        f"  Novice: {c['novice']}\n  Competent: {c['competent']}\n  Expert: {c['expert']}"
        for c in criteria
    )


def load_program(state):
    items = [{**item, "index": i, "difficulty": DIFFICULTY_MAP.get(item.get("task_type", ""), 2)}
             for i, item in enumerate(state["item_bank"])]
    random.shuffle(items)
    domain_rubrics = _build_domain_rubrics(state["rubrics"], state["assessments"])
    domains = [d["name"] for d in state["framework"].get("domains", [])]
    proficiency = {d: {"score": 0.0, "items_count": 0, "level": "untested"} for d in domains}
    return {"items_remaining": items, "domain_rubrics": domain_rubrics,
            "domain_proficiency": proficiency, "current_item": None,
            "current_domain": "", "current_evaluation": None, "probed": False,
            "exam_complete": False, "passed": None, "result_summary": None}


def select_item(state):
    remaining = list(state["items_remaining"])
    prof = state["domain_proficiency"]
    untested = [d for d, p in prof.items() if p["items_count"] == 0]
    weak = [d for d, p in prof.items() if p["items_count"] > 0 and p["score"] < 2.0]
    priority = untested or weak
    chosen = None
    if priority:
        for item in remaining:
            if _extract_domain(item["competency_ref"]) in priority:
                chosen = item; break
    if not chosen and remaining:
        chosen = remaining[0]
    if not chosen:
        return {"exam_complete": True, "current_item": None}
    remaining.remove(chosen)
    return {"current_item": chosen, "current_domain": _extract_domain(chosen["competency_ref"]),
            "items_remaining": remaining, "current_evaluation": None, "probed": False}


def present_item(state):
    item = state["current_item"]
    if not item:
        return {"exam_complete": True}
    n_done = len(state.get("items_administered", []))
    n_total = n_done + len(state["items_remaining"]) + 1
    question = f"Question {n_done+1}/{n_total} — {state['current_domain']}\n\n{item['stem']}"
    response = interrupt({"type": "question", "content": question})
    return {"messages": [{"role": "agent", "content": question},
                         {"role": "learner", "content": response}]}


def evaluate_response(state):
    item = state["current_item"]
    criteria = state.get("domain_rubrics", {}).get(state["current_domain"], [])
    learner_response = next((m["content"] for m in reversed(state.get("messages", [])) if m["role"] == "learner"), "")
    rubric_text = _format_rubric_criteria(criteria) if criteria else "General criteria."
    structured_llm = llm.with_structured_output(EvaluationResult)
    result = structured_llm.invoke([
        SystemMessage(content=(
            f"Score the response against rubric criteria.\n\n"
            f"Item: {item['stem']}\nModel Answer: {item['model_answer']}\n"
            f"Scoring Notes: {item['scoring_notes']}\nRubric:\n{rubric_text}\n\n"
            f"Score each criterion 1-3. Set confidence to 'clear' or 'borderline'.")),
        HumanMessage(content=f"Response:\n{learner_response}")])
    return {"current_evaluation": result.model_dump()}


def probe_or_score(state):
    probe_q = state["current_evaluation"].get("probe_question", "Can you elaborate?")
    follow_up = interrupt({"type": "probe", "content": probe_q})
    item = state["current_item"]
    criteria = state.get("domain_rubrics", {}).get(state["current_domain"], [])
    orig = next((m["content"] for m in state.get("messages", []) if m["role"] == "learner"), "")
    rubric_text = _format_rubric_criteria(criteria) if criteria else "General criteria."
    structured_llm = llm.with_structured_output(EvaluationResult)
    result = structured_llm.invoke([
        SystemMessage(content=(
            f"Re-evaluate with follow-up context.\nItem: {item['stem']}\n"
            f"Model Answer: {item['model_answer']}\nRubric:\n{rubric_text}\n"
            f"Original: {orig}\nProbe: {probe_q}")),
        HumanMessage(content=f"Follow-up:\n{follow_up}")])
    return {"messages": [{"role": "agent", "content": probe_q},
                         {"role": "learner", "content": follow_up}],
            "current_evaluation": result.model_dump(), "probed": True}


def update_proficiency(state):
    ev = state["current_evaluation"]
    domain = state["current_domain"]
    score = ev["weighted_score"]
    prof = dict(state["domain_proficiency"])
    cur = prof.get(domain, {"score": 0.0, "items_count": 0, "level": "untested"})
    n = cur["items_count"] + 1
    avg = ((cur["score"] * cur["items_count"]) + score) / n
    level = "novice" if avg < 1.7 else ("competent" if avg < 2.5 else "expert")
    prof[domain] = {"score": round(avg, 2), "items_count": n, "level": level}
    return {"domain_proficiency": prof,
            "items_administered": [{"domain": domain, "score": score, "feedback": ev.get("feedback", "")}],
            "messages": [{"role": "agent", "content": f"Feedback: {ev.get('feedback', '')}"}]}


def determine_result(state):
    prof = state["domain_proficiency"]
    tested = {d: p for d, p in prof.items() if p["items_count"] > 0}
    overall = sum(p["score"] for p in tested.values()) / max(len(tested), 1)
    passed = all(p["score"] >= 2.0 for p in tested.values()) or \
             (overall >= 2.0 and all(p["score"] >= 1.5 for p in tested.values()))
    structured_llm = llm.with_structured_output(ExamResultSummary)
    result = structured_llm.invoke([
        SystemMessage(content=f"Summarise exam results.\nDomains: {json.dumps({d: p for d, p in prof.items()}, indent=2)}\n"
                      f"Overall: {overall:.2f}/3.00, Passed: {passed}"),
        HumanMessage(content="Generate result summary.")])
    rd = result.model_dump()
    rd.update({"overall_score": round(overall, 2), "passed": passed, "domain_breakdown": prof})
    return {"exam_complete": True, "passed": passed, "result_summary": rd,
            "messages": [{"role": "agent", "content": f"Exam Complete\n\n{result.summary}\n\nRecommendation: {result.recommendation}"}]}


print("All 7 node functions defined.")

All 7 node functions defined.


## 5. Build the Graph

The graph uses conditional edges after evaluation (borderline → probe, clear → update)
and after proficiency update (more items → loop, done → result).

In [5]:
def after_evaluate(state):
    ev = state.get("current_evaluation", {})
    if ev.get("confidence") == "borderline" and not state.get("probed", False):
        return "probe_or_score"
    return "update_proficiency"

def after_update(state):
    if not state["items_remaining"]:
        return "determine_result"
    prof = state["domain_proficiency"]
    domains = list(prof.keys())
    all_tested = all(prof[d]["items_count"] > 0 for d in domains)
    if all_tested and len(state.get("items_administered", [])) >= len(domains):
        uncertain = [d for d in domains if prof[d]["items_count"] < 2
                     and any(_extract_domain(it["competency_ref"]) == d for it in state["items_remaining"])]
        if not uncertain:
            return "determine_result"
    return "select_item"


memory = MemorySaver()
builder = StateGraph(ExamState)

builder.add_node("load_program", load_program)
builder.add_node("select_item", select_item)
builder.add_node("present_item", present_item)
builder.add_node("evaluate_response", evaluate_response)
builder.add_node("probe_or_score", probe_or_score)
builder.add_node("update_proficiency", update_proficiency)
builder.add_node("determine_result", determine_result)

builder.add_edge(START, "load_program")
builder.add_edge("load_program", "select_item")
builder.add_edge("select_item", "present_item")
builder.add_edge("present_item", "evaluate_response")
builder.add_conditional_edges("evaluate_response", after_evaluate, {
    "probe_or_score": "probe_or_score",
    "update_proficiency": "update_proficiency",
})
builder.add_edge("probe_or_score", "update_proficiency")
builder.add_conditional_edges("update_proficiency", after_update, {
    "select_item": "select_item",
    "determine_result": "determine_result",
})
builder.add_edge("determine_result", END)

exam_graph = builder.compile(checkpointer=memory)
print("Exam graph compiled.")

Exam graph compiled.


## 6. Run the Exam (Simulated)

We start the exam and simulate learner responses. Each `interrupt()` pauses the graph;
we resume with `Command(resume=response)`.

In [6]:
config = {"configurable": {"thread_id": "demo-exam-001"}}

initial_state = {
    "program_id": "demo",
    "learner_id": "demo-learner",
    "program_name": "AI Champion",
    "item_bank": program_data["item_bank"],
    "rubrics": program_data["rubrics"],
    "framework": program_data["competency_framework"],
    "assessments": program_data["assessments"],
    "items_remaining": [],
    "domain_rubrics": {},
    "current_item": None,
    "current_domain": "",
    "current_evaluation": None,
    "probed": False,
    "messages": [],
    "items_administered": [],
    "domain_proficiency": {},
    "exam_complete": False,
    "passed": None,
    "result_summary": None,
}

exam_graph.invoke(initial_state, config=config)

state = exam_graph.get_state(config)
print(f"Graph paused at: {state.next}")
interrupt_val = state.tasks[0].interrupts[0].value
print(f"\nFirst question:\n{interrupt_val['content']}")

Graph paused at: ('present_item',)

First question:
Question 1/10 — Connectors/Integrations

Your enterprise is planning to deploy an AI agent across multiple communication channels, including web, mobile, and social media platforms. Describe the strategic considerations and steps you would take to ensure successful deployment and optimal performance across these channels.


In [7]:
# Simulate a competent-level response
simulated_response = (
    "To design a conversational AI agent, I would start by mapping out the key user intents "
    "and creating a topic hierarchy in Copilot Studio. Each intent gets its own topic with "
    "trigger phrases and a response flow. I would use entities to extract important information "
    "from user messages and route to appropriate topics. For managing multiple intents, I'd "
    "implement a disambiguation flow that presents options when the system isn't confident "
    "about the user's intent. To adjust based on feedback, I'd review the analytics dashboard "
    "to identify topics with high abandonment rates and refine the trigger phrases and responses."
)

exam_graph.invoke(Command(resume=simulated_response), config=config)

state = exam_graph.get_state(config)
print(f"Graph paused at: {state.next}")
if state.next:
    interrupt_val = state.tasks[0].interrupts[0].value
    msg_type = interrupt_val.get('type', 'unknown')
    print(f"Type: {msg_type}")
    print(f"\nContent:\n{interrupt_val['content']}")
else:
    print("Exam complete!")
    print(json.dumps(state.values.get("result_summary"), indent=2))

Graph paused at: ('present_item',)
Type: question

Content:
Question 2/10 — Conversational Design

You are tasked with analyzing user engagement data from an AI agent deployed in a retail environment. Describe the process you would follow to identify trends and make recommendations for improving the agent's conversational design.


/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluationResult(criterio...ocial media platforms?'), input_type=EvaluationResult])
  return self.__pydantic_serializer__.to_python(


## 7. Continue the Exam Loop

We can continue responding until the exam completes. Let's run through
a few more items with simulated responses.

In [8]:
simulated_answers = [
    "I would configure the agent to use OAuth 2.0 for authentication and set up DLP policies "
    "to prevent data leakage. Performance monitoring would involve tracking response times "
    "and user satisfaction metrics through the analytics dashboard.",
    
    "For deploying across channels, I'd start with Teams as the primary channel, then extend "
    "to the website via a web chat widget. Each channel would have tailored greeting messages "
    "and I'd use the channel-specific settings to optimise the experience for each platform.",
    
    "I would integrate with the CRM using custom connectors, mapping the agent's conversation "
    "data to CRM fields. The integration would use REST APIs with proper error handling and "
    "retry logic to ensure reliability.",
]

for i, answer in enumerate(simulated_answers):
    state = exam_graph.get_state(config)
    if not state.next:
        print("Exam already complete.")
        break
    
    interrupt_val = state.tasks[0].interrupts[0].value
    msg_type = interrupt_val.get("type", "unknown")
    
    if msg_type == "probe":
        print(f"--- Probe: {interrupt_val['content'][:80]}...")
    else:
        print(f"--- Question: {interrupt_val['content'][:80]}...")
    
    exam_graph.invoke(Command(resume=answer), config=config)
    state = exam_graph.get_state(config)
    prof = state.values.get("domain_proficiency", {})
    for d, p in prof.items():
        if p["items_count"] > 0:
            print(f"  {d}: {p['score']:.2f} ({p['level']})")
    print()

--- Question: Question 2/10 — Conversational Design

You are tasked with analyzing user engage...


/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluationResult(criterio....', probe_question=None), input_type=EvaluationResult])
  return self.__pydantic_serializer__.to_python(


  Conversational Design: 1.00 (novice)
  Connectors/Integrations: 1.00 (novice)

--- Question: Question 3/10 — Agent Creation & Configuration

You are designing an AI agent th...
  Agent Creation & Configuration: 3.00 (expert)
  Conversational Design: 1.00 (novice)
  Connectors/Integrations: 1.00 (novice)

--- Question: Question 4/10 — Security/Governance

You are tasked with developing a governance...
  Agent Creation & Configuration: 3.00 (expert)
  Conversational Design: 1.00 (novice)
  Connectors/Integrations: 1.00 (novice)
  Security/Governance: 3.00 (expert)



## 8. Inspect Final Results

In [9]:
# Keep responding until the exam finishes
fallback_answer = (
    "I would follow best practices for this area, ensuring proper configuration, "
    "testing, and documentation. I'd also review the official Microsoft documentation "
    "for the latest guidance on implementation."
)

for _ in range(20):
    state = exam_graph.get_state(config)
    if not state.next:
        break
    exam_graph.invoke(Command(resume=fallback_answer), config=config)

final_state = exam_graph.get_state(config)
result = final_state.values.get("result_summary")

if result:
    print(f"Passed: {result['passed']}")
    print(f"Overall Score: {result['overall_score']:.2f}/3.00")
    print(f"\nSummary: {result['summary']}")
    print(f"\nRecommendation: {result['recommendation']}")
    print(f"\nDomain Breakdown:")
    for d, p in result.get("domain_breakdown", {}).items():
        print(f"  {d}: {p['score']:.2f} ({p['level']}) — {p['items_count']} items")
else:
    print("Exam did not complete.")

/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluationResult(criterio...rations in the design?'), input_type=EvaluationResult])
  return self.__pydantic_serializer__.to_python(
/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=EvaluationResult(criterio... for this integration?'), input_type=EvaluationResult])
  return self.__pydantic_serializer__.to_python(
/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUn

Passed: False
Overall Score: 1.61/3.00

Summary: The exam results indicate varying levels of proficiency across different domains. The candidate demonstrated competence in 'Agent Creation & Configuration' and 'Security/Governance', with scores of 2.12 and 2.17 respectively. However, the candidate showed novice-level understanding in 'Conversational Design' and 'Connectors/Integrations', scoring 1.0 and 1.17 respectively.

Recommendation: To improve overall performance, it is recommended that the candidate focuses on enhancing skills in 'Conversational Design' and 'Connectors/Integrations'. Additional training or study in these areas could help achieve a passing score in future assessments.

Domain Breakdown:
  Agent Creation & Configuration: 2.12 (competent) — 2 items
  Conversational Design: 1.00 (novice) — 2 items
  Connectors/Integrations: 1.17 (novice) — 2 items
  Security/Governance: 2.17 (competent) — 2 items


/Users/josephmata/Documents/Cloud_backup/CertOps/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ExamResultSummary(passed=...in future assessments."), input_type=ExamResultSummary])
  return self.__pydantic_serializer__.to_python(


## Recap

| Feature | Implementation |
|---------|---------------|
| **Adaptive selection** | Prioritises untested/weak domains, stops when coverage is sufficient |
| **LLM-as-Judge** | GPT-4o evaluates free-text responses against rubrics with structured output |
| **Conversational probing** | Borderline responses trigger targeted follow-up questions |
| **Session persistence** | `MemorySaver` (local) / `PostgresSaver` (production) checkpointing |
| **Pass/fail logic** | Competent (2.0) in all domains, or 2.0 overall with no domain below 1.5 |

### Production Implementation

These patterns are implemented in the production CertOps application — the FastAPI backend
hosts the exam graph at `/exam/*` endpoints, and the Next.js frontend provides a chat-based
assessment interface with real-time domain proficiency tracking.